In [3]:
import jax
import jax.numpy as jnp

def netA(params, x):
    w1, b1 = params["A"]
    return jnp.dot(x, w1) + b1

def netB(params, x):
    w2, b2 = params["B"]
    return jnp.dot(x, w2) + b2

def forward_full(params, x, frozen_B:bool):
    out_a = netA(params, x)
    out_b = netB(params, x)
    if frozen_B:
        out_b = jax.lax.stop_gradient(out_b)
    y = jnp.concatenate([out_a, out_b], axis=-1)
    return y

def loss_fn(params, x, frozen_B):
    y = forward_full(params, x, frozen_B)
    return jnp.sum(jnp.abs(y))

key = jax.random.PRNGKey(42)
params = {
    "A": (jax.random.normal(key, (3,2)), jnp.zeros(2)),
    "B": (jax.random.normal(key, (3,2)), jnp.zeros(2)),
}
x = jnp.array([[1.0, 2.0, 3.0]])

# --------------------
# static_argnums=2：第3个参数 frozen_B 是静态Python值
# --------------------
grad_loss = jax.grad(loss_fn)

print("=== 不冻结 ===")
g1 = grad_loss(params, x, frozen_B=False)
print("g A w1:\n", g1["A"][0])
print("g B w2:\n", g1["B"][0])

print("\n=== 冻结B ===")
g2 = grad_loss(params, x, frozen_B=True)
print("g A w1:\n", g2["A"][0])
print("g B w2:\n", g2["B"][0])

# -------- make_jaxpr：指定 static_argnums！！ --------
print("\n======== jaxpr 不冻结B ========")
print(jax.make_jaxpr(grad_loss, static_argnums=2)(params, x, False))

print("\n======== jaxpr 冻结B ========")
print(jax.make_jaxpr(grad_loss, static_argnums=2)(params, x, True))


=== 不冻结 ===
g A w1:
 [[1. 1.]
 [2. 2.]
 [3. 3.]]
g B w2:
 [[1. 1.]
 [2. 2.]
 [3. 3.]]

=== 冻结B ===
g A w1:
 [[1. 1.]
 [2. 2.]
 [3. 3.]]
g B w2:
 [[0. 0.]
 [0. 0.]
 [0. 0.]]

======== jaxpr 不冻结B ========
{ lambda ; a:f32[3,2] b:f32[2] c:f32[3,2] d:f32[2] e:f32[1,3]. let
    f:f32[1,2] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] e a
    g:f32[1,2] = broadcast_in_dim[broadcast_dimensions=(1,)] b
    h:f32[1,2] = add f g
    i:f32[1,2] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] e c
    j:f32[1,2] = broadcast_in_dim[broadcast_dimensions=(1,)] d
    k:f32[1,2] = add i j
    l:f32[1,4] = concatenate[dimension=1] h k
    m:f32[1,4] = abs l
    n:bool[1,4] = ge l 0.0:f32[]
    _:f32[] = reduce_sum[axes=(0, 1) out_sharding=None] m
    o:f32[1,4] = broadcast_in_dim 1.0:f32[]
    p:f32[1,4] = broadcast_in_dim 0.0:f32[]
    q:f32[1,4] = select_n n o p
    r:f32[1,4] = select_n 